In [0]:
# Cell 1 - Setup
storage_account_name = "stkalshianalytics"
storage_account_key = dbutils.secrets.get("kalshi-scope", "adls-storage-key")

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net"
gold_path = f"abfss://gold@{storage_account_name}.dfs.core.windows.net"

sql_server = "sql-edgar-analytics.database.windows.net"
sql_database = "edgar-analytics"
sql_user = "edgaradmin"
sql_password = dbutils.secrets.get("kalshi-scope", "sql-password")

jdbc_url = f"jdbc:sqlserver://{sql_server}:1433;database={sql_database};user={sql_user};password={sql_password};encrypt=true;trustServerCertificate=false"

print("Ready")


In [0]:
# Cell 2 - Build and Write Gold
from pyspark.sql.functions import col, when, substring, current_timestamp, from_utc_timestamp, round as spark_round, to_date, datediff, month, year, current_date, coalesce
from datetime import datetime, timezone

gold_start = datetime.now(timezone.utc)  # FIX: moved to top

df_silver = spark.read.format("delta").load(f"{silver_path}/kalshi_markets")

def map_category(prefix_6_col, prefix_long_col):
    return (
        when(prefix_6_col.isin("KXNFLD","KXNFLP","KXNFLT","KXNFLO","KXNFLN","KXNFLA","KXNFLM"), "NFL")
        .when(prefix_6_col == "KXSB-", "NFL")
        .when(prefix_6_col.isin("KXNBA1","KXNBA2","KXNBA3","KXNBAP","KXNBAR","KXNBAS","KXNBAT","KXNBAW","KXNBAA","KXNBAC","KXNBAD","KXNBAE","KXNBAF","KXNBAG","KXNBAM","KXNBAN","KXNBAB","KXNBA-"), "NBA")
        .when(prefix_6_col.isin("KXMLBH","KXMLBT","KXMLBS","KXMLBW","KXMLBN","KXMLBA","KXMLBM","KXMLBF","KXMLBK","KXMLBR","KXMLBE","KXMLBL","KXMLBP","KXMLBG","KXMLB-","KXMLBB"), "MLB")
        .when(prefix_6_col.isin("KXNHLT","KXNHLS","KXNHLP","KXNHLR","KXNHLA","KXNHLC","KXNHLG","KXNHLH","KXNHLV","KXNHLW","KXNHLE","KXNHLM","KXNHLN","KXNHL-"), "NHL")
        .when(prefix_long_col.startswith("KXMARMAD"), "March Madness")
        .when(prefix_6_col == "KXNCAA", "NCAA")
        .when(prefix_long_col.startswith("KXIPLGAME"), "Cricket/IPL")
        .when(prefix_6_col.isin("KXIPLGA"), "Cricket/IPL")
        .when(prefix_6_col.isin("KXSERI","KXBUND","KXLIGU","KXEPL1","KXEPLG","KXEPLR","KXEPLT","KXEPLS","KXEPLB","KXUCL","KXUECL","KXWCRO","KXWCGA","KXWCGR","KXWCGO","KXWCSQ","KXFIFA","KXUCLG","KXUCLRO","KXUCLW", "KXLAI"), "Soccer")
        .when(prefix_6_col.isin("KXUE-A","KXUE-B","KXUE-C","KXUE-F","KXUE-G","KXUE-I","KXUE-J","KXUE-M","KXUE-R","KXUE-U","KXUEL-"), "Soccer")
        .when(prefix_6_col.isin("KXSOCC","KXUCLS","KXUCLT","KXUCL1","KXUCL-","KXUCLR","KXUCLB","KXUCLF"), "Soccer")
        .when(prefix_6_col.isin("KXPGAT","KXPGAR","KXPGAM","KXPGAH","KXLPGA","KXPGAC"), "Golf")
        .when(prefix_6_col.isin("KXATPS","KXATPC","KXATPM","KXATPG","KXATP1","KXWTAM","KXWTAG","KXWTIM","KXWTAS"), "Tennis")
        .when(prefix_6_col.isin("KXCS2M","KXCS2G","KXCS2T","KXVALO","KXLOLG","KXLOLM","KXLOLT","KXR6GA"), "Esports")
        .when(prefix_6_col.isin("KXUFCF","KXUFCL","KXUFCW","KXUFCB","KXUFCH","KXUFCM"), "UFC/MMA")
        .when(prefix_6_col == "KXWNBA", "WNBA")
        .when(prefix_6_col.isin("KXF1-2","KXF1CO","KXF1CH","KXF1OC"), "Formula 1")
        .when(prefix_6_col == "KXAFLG", "Australian Football")
        .when(prefix_6_col.isin("KXFIBA","KXWBCF","KXWBCB","KXWBCM","KXWBCW","KXWBCH","KXWBCL"), "Other Sports")
        .when(prefix_6_col.isin("KXCBAG","KXBALL","KXTOUR","KXRUGB","KXBOXL","KXBOXI","KXINDY","KXMOTO","KXU3MA","KXU3-2"), "Other Sports")
        .when(prefix_6_col.isin("KXSENA","KXTRUM","KXPRES","KXVOTE","KXBILL","KXCONG","KXVPRE","SENATE","HOUSEC","HOUSEI","HOUSEM","HOUSEN","HOUSEO","HOUSEP","HOUSEV","HOUSEW","HOUSEF","HOUSEA","KXGOVM","KXGOVA","KXGOVC","KXGOVN","KXGOVT","KXGOVI","KXGOVK","KXGOVV","KXGOVW","KXGOVG","KXGOVP","KXGOVR","KXGOVS","KXGOVO","KXGOVF","KXPRIM","KXGAPR","KXKYPR","KXORPR","KXALPR","KXIDPR","KXPAPR","KXLAPR","KXMDPR","KXNVPR","KXOHPR","KXTXPR","KXWVPR","KXBRPR","GOVPAR"), "Politics/Government")
        .when(prefix_long_col.startswith("KXLAYOFFSYINFO"), "Economics/Markets")
        .when(prefix_long_col.startswith("KXFEDDECISION"), "Economics/Markets")
        .when(prefix_long_col.startswith("KXCITRINI"), "Economics/Markets")
        .when(prefix_6_col.isin("KXECON","KXFED-","KXFEDD","KXFEDM","KXFEDE","KXFEDC","KXFEDG","KXFEDL","KXFOMC","KXRATE","KXCPI-","KXCPIC","KXCPIY","KXGDPN","KXGDPY","KXGDP-","KXUSTY","KXADP-","KXPAYR","KXJOBL","KXNASD","KXNASC","KXINX-","KXINXU","KXINXY","KXINXM","KXINXP","KXSP50","KXBOND","KXEOWE","KXEOTR"), "Economics/Markets")
        .when(prefix_6_col.isin("KXBTC-","KXBTCD","KXBTCM","KXBTCY","KXBTCH","KXBTC1","KXBTC2","KXBTCR","KXBTCV","KXETH-","KXETHD","KXETHM","KXETHY","KXETH1","KXDOGE","KXBNB-","KXBNBD","KXXRP-","KXXRPD","KXXRPM","KXXRP1","KXSHIB","KXSOL1","KXSOL2","KXSOLM","KXDASH","KXCRYP","KXTOKE"), "Crypto")
        .when(prefix_6_col.isin("KXGOLD","KXSILV","KXCOPP","KXNATG","KXWTI-","KXWTIW","KXOILR","KXNGAS"), "Commodities")
        .when(prefix_6_col.isin("KXMUSK","KXTESL","KXMETA","KXAISP","KXAIST","KXLLM1","KXGPTC","KXAAPL","KXSPAC","KXGROK","KXJPMO","KXJPMC"), "Tech/AI")
        .when(prefix_6_col.isin("KXALBU","KXOSCA","KXMOVI","KXSNLH","KXSNLM","KXSNL-","KXPODC","KXTVSH","KXBIGB","KXSURV","KXACTO","KXROLL","KXVENU","KXMENW","KXGAME","KXSUPE","KXBREN","KXFEAT","KXLOVE","BEYONC","KXSWIF","KXTAYL","KXSONG","KXGRAM"), "Entertainment")
        .when(prefix_6_col.isin("KXNEXT","KXLEAD","KXRANK","KXTEAM","KXHYPE","KXROLE","KXSOLD","KXSOLE","KXHIGH","KXLALI","KXSTAR","KXTOPA","KXTOPS","KXTOPM","KXTOPC","KXTOPB","KXTOP3","KXTOPP","KX1SON","KX10SO","KX20SO","KX1ALB","KXPERF","KXSPOT","KXRECO","KXLOSE"), "Pop Culture/Rankings")
        .when(prefix_6_col.isin("KXTEMP","KXRAIN","KXTORN"), "Weather")
        .when(prefix_6_col.isin("KXBRAS","KXEURO","KXINTL","KXPUTI","KXISRA","KXUKRE","KXIRAN","KXVENE","KXBRAZ","KXTHAI","KXMARM","KXTURK","KXGREE","KXBELG","KXDENM","KXFINL","KXSPAI","KXSWED","KXITAL","KXPOLA","KXNIGE","KXMALA","KXKENY","KXGHAN","KXZAMB","KXAFRI","KXLATV","KXPERU","KXARGE","KXMONG","KXMEXC","KXMOLD","KXBULG","KXSLOV","KXICER","KXDOMI","KXCANA","KXCOLO","KXQUEB","KXWALE","KXSCOT","KXNEWZ","KXAUST","KXJAPA","KXSEOU","KXOAIA"), "International")
        .when(prefix_6_col == "KXHOUS", "Housing/Real Estate")
        .when(prefix_6_col.isin("KXIPOO","KXIPOA","KXIPOB","KXIPOC","KXIPOD","KXIPOF","KXIPOG","KXIPOR","KXIPOS","KXIPOW","KXIPO-"), "IPOs")
        .when(prefix_6_col.isin("KXNETF","KXAAAG","KXRTX5","KXH100","KXH200","KXB200","KXA100"), "ETFs/Stocks")
        .when(prefix_6_col.isin("KXRT-D","KXRT-M","KXRT-S","KXRT-Y","KXRT-A","KXRT-O"), "Real-Time Markets")
        .when(prefix_6_col.isin("KXLOWT","KXCBDE","KXMEDI","KXPCEC","KXSECP","KXSCOU","KXATTY","KXFDAA"), "Policy/Legal")
        .otherwise("Other")
    )

prefix_6 = substring(col("event_ticker"), 1, 6)
prefix_long = substring(col("event_ticker"), 1, 15)

df_gold = df_silver \
    .withColumn("prefix", prefix_6) \
    .withColumn("category", coalesce(col("kalshi_category"), map_category(prefix_6, prefix_long))) \
    .withColumn("implied_prob_yes", spark_round((col("yes_ask") + col("yes_bid")) / 2, 4)) \
    .withColumn("implied_prob_no", spark_round((col("no_ask") + col("no_bid")) / 2, 4)) \
    .withColumn("spread", spark_round(col("yes_ask") - col("yes_bid"), 4)) \
    .withColumn("is_single_market", col("mve_collection_ticker").isNull()) \
    .withColumn("has_liquidity", col("liquidity") > 0) \
    .withColumn("market_quality",
        when(col("liquidity") > 1000, "High")
        .when(col("liquidity") > 100, "Medium")
        .when(col("liquidity") > 0, "Low")
        .otherwise("None")
    ) \
    .withColumn("close_date", to_date(col("close_time"))) \
    .withColumn("days_until_close", datediff(col("close_date"), current_date())) \
    .withColumn("close_month", month(col("close_time"))) \
    .withColumn("close_year", year(col("close_time"))) \
    .withColumn("gold_updated_at", from_utc_timestamp(current_timestamp(), "America/New_York")) \
    .select(
        "ticker", "title", "event_ticker", "prefix", "category", "status",
        "yes_ask", "yes_bid", "no_ask", "no_bid", "last_price",
        "implied_prob_yes", "implied_prob_no", "spread",
        "liquidity", "volume_24h", "volume_total", "open_interest",
        "market_quality", "is_single_market", "has_liquidity",
        "close_time", "close_date", "days_until_close", "close_month", "close_year",
        "is_provisional", "mve_collection_ticker", "no_sub_title", "yes_sub_title",
        "result", "can_close_early", "gold_updated_at"
    )

# FIX: count once and reuse
gold_count = df_gold.count()
print(f"Gold row count: {gold_count}")
df_gold.groupBy("category").count().orderBy("count", ascending=False).show(30, truncate=False)

(df_gold.write
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "kalshi_gold.fact_markets")
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .mode("overwrite")
    .save()
)

print("Gold table written to Azure SQL successfully")


In [0]:
# Cell 2b - History Table Append
from pyspark.sql.functions import current_date

df_history = df_gold.filter(col("category") == "Sports").select(
    "ticker",
    "title",
    "category",
    "implied_prob_yes",
    "implied_prob_no",
    "yes_bid",
    "yes_ask",
    "no_bid",
    "no_ask",
    "last_price",
    "spread",
    "volume_24h",
    "volume_total",
    "liquidity",
    "open_interest",
    "market_quality",
    "days_until_close"
).withColumn("snapshot_date", current_date())

# Append today's snapshot
(df_history.write
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "kalshi_gold.fact_markets_history")
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .mode("append")
    .save())

history_count = df_history.count()
print(f"History snapshot appended: {history_count} rows")

# Clean up rows older than 7 days using Java driver directly
driver_manager = spark._jvm.java.sql.DriverManager
connection = driver_manager.getConnection(
    jdbc_url,
    sql_user,
    sql_password
)
statement = connection.createStatement()
rows_deleted = statement.executeUpdate(
    "DELETE FROM kalshi_gold.fact_markets_history WHERE snapshot_date < CAST(DATEADD(day, -90, GETDATE()) AS DATE)"
)
statement.close()
connection.close()

print(f"History cleanup complete: {rows_deleted} old rows removed")

In [0]:
# Cell 3 - Odds API
import requests
from datetime import datetime, timezone

odds_api_key = dbutils.secrets.get(scope="kalshi-scope", key="odds-api-key")

SPORT_KEYS = [
    "soccer_epl",
    "soccer_uefa_champs_league",
    "soccer_spain_la_liga",
    "soccer_germany_bundesliga",
    "soccer_italy_serie_a",
    "soccer_france_ligue_one",
    "soccer_usa_mls",
    "soccer_mexico_ligamx",
    "soccer_fifa_world_cup",
    "soccer_uefa_european_championship",
    "soccer_conmebol_copa_america",
    "soccer_international_friendlies",
    "soccer_england_fa_cup",
    "soccer_spain_copa_del_rey",
    "soccer_germany_dfb_pokal",
    "soccer_uefa_europa_league",
    "soccer_uefa_europa_conference_league",
]

REGIONS = "us"
MARKETS = "h2h,spreads,totals"
ODDS_FORMAT = "decimal"

all_rows = []
games_per_league = []

for SPORT_KEY in SPORT_KEYS:
    url = f"https://api.the-odds-api.com/v4/sports/{SPORT_KEY}/odds"
    params = {
        "apiKey": odds_api_key,
        "regions": REGIONS,
        "markets": MARKETS,
        "oddsFormat": ODDS_FORMAT
    }
    
    response = requests.get(url, params=params)
    
    if response.status_code != 200:
        print(f"⚠️ {SPORT_KEY}: Error {response.status_code} - {response.text}")
        continue
    
    data = response.json()
    games_count = len(data)
    games_per_league.append((SPORT_KEY, games_count))
    print(f"✅ {SPORT_KEY}: {games_count} games")
    
    for game in data:
        commence = game.get("commence_time", "")
        home = game.get("home_team", "")
        away = game.get("away_team", "")
        
        for bookmaker in game.get("bookmakers", []):
            bookie_name = bookmaker.get("key", "")
            last_update = bookmaker.get("last_update", "")
            
            for market in bookmaker.get("markets", []):
                market_key = market.get("key", "")
                
                for outcome in market.get("outcomes", []):
                    all_rows.append({
                        "sport_key": SPORT_KEY,
                        "game_id": game.get("id", ""),
                        "commence_time": commence,
                        "home_team": home,
                        "away_team": away,
                        "bookmaker": bookie_name,
                        "market": market_key,
                        "outcome_name": outcome.get("name", ""),
                        "point": outcome.get("point", None),      # FIX: spread/total line
                        "price": outcome.get("price", None),       # FIX: actual odds value
                        "last_update": last_update,
                        "pulled_at": datetime.utcnow().isoformat()
                    })

print(f"\nTotal rows pulled: {len(all_rows)}")
for league, count in games_per_league:
    print(f"  {league}: {count}")

if all_rows:
    df_odds = spark.createDataFrame(all_rows)
    (df_odds.write
        .format("jdbc")
        .option("url", jdbc_url)
        .option("dbtable", "kalshi_gold.odds_api_raw")
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
        .mode("overwrite")
        .save()
    )
    print("✅ All soccer odds written to kalshi_gold.odds_api_raw")
else:
    print("⚠️ No odds data pulled - nothing written to SQL")


In [0]:
# Cell 4 - Gold Metadata Logging
from datetime import datetime, timezone
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

# FIX: duration captures full gold processing time
gold_duration = (datetime.now(timezone.utc) - gold_start).total_seconds()

metadata_schema = StructType([
    StructField("layer", StringType(), False),
    StructField("source", StringType(), False),
    StructField("records_added", IntegerType(), False),
    StructField("row_count", IntegerType(), False),
    StructField("duration_sec", DoubleType(), False),
    StructField("status", StringType(), False),
    StructField("error_message", StringType(), True),
    StructField("last_refresh", TimestampType(), False)
])

metadata_df = spark.createDataFrame(
    [{
        "layer": "gold",
        "source": "Databricks (03_gold_kalshi)",
        "records_added": int(gold_count),
        "row_count": int(gold_count),
        "duration_sec": round(gold_duration, 2),
        "status": "success",
        "error_message": None,
        "last_refresh": datetime.now(timezone.utc)
    }],
    schema=metadata_schema
)

(metadata_df.write
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "kalshi_gold.pipeline_metadata")
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .mode("append")
    .save())

print(f"Gold complete: {gold_count} rows in {gold_duration:.1f}s")
